# Day 7 — Hugging Face Foundations
## 30 Days of AI: From NLP to LLMs

---

On Day 6 you built a Transformer from scratch — every matrix multiply,
every attention head, every positional encoding written by hand.
You now understand exactly what is happening inside the most important
architecture in modern AI.

The next question is: how do practitioners actually use these models?
Nobody re-implements GPT-3 from scratch for each project. The answer
is Hugging Face — the GitHub of machine learning.

Hugging Face hosts over 900,000 models, 200,000 datasets, and the
most widely used NLP library in the world: `transformers`. Today
you learn what Hugging Face is, why it exists, and how to use
its three core building blocks: pipelines, tokenizers, and models.

---

### What You Will Learn Today

- What Hugging Face is and why the AI community built it
- The Hub — how models, datasets, and spaces are organized
- Pipelines — one-line inference for 20+ NLP tasks
- Tokenizers — how raw text becomes model input
- AutoModel and AutoTokenizer — loading any model in two lines
- The difference between encoder, decoder, and seq2seq models on the Hub
- How to read a model card and choose the right model for your task

### Goal by End of Day

Run sentiment analysis, named entity recognition, summarization,
and question answering using Hugging Face pipelines. Understand
what a tokenizer does and inspect its output. Load a raw model
and run a forward pass manually.

In [17]:
## Run once to install dependencies
#!pip install transformers datasets torch sentencepiece -q
import warnings
warnings.filterwarnings('ignore')
import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline

print('PyTorch version  :', torch.__version__)
print('Device available :', 'GPU' if torch.cuda.is_available() else 'CPU')

import transformers
print('Transformers version :', transformers.__version__)

PyTorch version  : 2.10.0+cpu
Device available : CPU
Transformers version : 4.46.3


---

## Part 1 — What Is Hugging Face and Why Does It Exist?

Before Hugging Face, using a pretrained language model meant:

```
Step 1  :  Find the original research paper
Step 2  :  Hunt for the author's GitHub repository
Step 3  :  Figure out their custom data format
Step 4  :  Download weights from Google Drive or AWS (if still working)
Step 5  :  Adapt their training code to your task
Step 6  :  Debug incompatible PyTorch/TensorFlow versions
Step 7  :  Give up and implement it yourself
```

This was the state of NLP as recently as 2019. Every team reinvented
the same wheels. Hugging Face solved this by creating a unified
interface and a central hub.

### The Three Pillars of Hugging Face

```
1. The Hub (huggingface.co)
   → Central repository for models, datasets, and demo apps (Spaces)
   → Any model can be loaded with one identifier string
   → Model cards document what each model does, its limitations, biases

2. The transformers Library
   → Unified Python API for 200+ model architectures
   → Works with PyTorch, TensorFlow, and JAX
   → From one-line pipelines to low-level forward passes

3. The datasets Library
   → 60,000+ datasets, all loadable with one line
   → Memory-mapped — can stream datasets larger than RAM
   → Built-in preprocessing, splitting, and formatting utilities
```

### The Hub Model Taxonomy

Models on the Hub are organized by architecture type:

```
Encoder-only (BERT family)
  → Best for : classification, NER, question answering, embeddings
  → Examples : bert-base-uncased, roberta-base, distilbert-base
  → Sees full context in both directions — left AND right of each token

Decoder-only (GPT family)
  → Best for : text generation, completion, chat, code
  → Examples : gpt2, mistralai/Mistral-7B, meta-llama/Llama-2-7b
  → Causal attention — each token only sees tokens to its LEFT

Encoder-Decoder / Seq2Seq (T5 family)
  → Best for : translation, summarization, structured generation
  → Examples : t5-small, facebook/bart-large-cnn, google/flan-t5-base
  → Encoder reads full input, decoder generates token by token
```

This maps directly to what you learned on Day 6. Every model on the
Hub is one of these three patterns at its core.

---

## Part 2 — The Pipeline API

The pipeline is Hugging Face's highest-level abstraction. It bundles
tokenization, model inference, and output decoding into one call.

```
pipeline(task, model=None)

task   → what you want to do (string)
model  → which Hub model to use (optional — has a default)

Supported tasks include:
  'sentiment-analysis'         →  positive / negative with score
  'ner'                        →  named entity recognition
  'question-answering'         →  extract answer from context
  'summarization'              →  condense long text
  'translation_en_to_fr'       →  language translation
  'text-generation'            →  continue a prompt
  'fill-mask'                  →  predict masked token (BERT style)
  'zero-shot-classification'   →  classify without training
  'image-classification'       →  works with vision models too
  'automatic-speech-recognition' → transcribe audio
```

The first call downloads the model weights to a local cache.
Subsequent calls are instant — no re-download.

In [20]:
from transformers import pipeline

# ---------------------------------------------------------------
# Task 1 : Sentiment Analysis
# Default model : distilbert-base-uncased-finetuned-sst-2-english
# Architecture  : Encoder-only (DistilBERT — a compressed BERT)
# Trained on    : SST-2 (Stanford Sentiment Treebank)
# ---------------------------------------------------------------

sentiment = pipeline('sentiment-analysis')

texts = [
    'Hugging Face makes working with transformers incredibly easy.',
    'I spent three hours debugging a shape mismatch. I hate tensors.',
    'The model performance was acceptable but not remarkable.',
]

results = sentiment(texts)

print('Sentiment Analysis Results')
print('=' * 55)
for text, result in zip(texts, results):
    label = result['label']
    score = result['score']
    print(f'Text   : {text[:50]}...')
    print(f'Label  : {label}')
    print(f'Score  : {score:.4f}')
    print('-' * 55)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


Sentiment Analysis Results
Text   : Hugging Face makes working with transformers incre...
Label  : POSITIVE
Score  : 0.9982
-------------------------------------------------------
Text   : I spent three hours debugging a shape mismatch. I ...
Label  : NEGATIVE
Score  : 0.9987
-------------------------------------------------------
Text   : The model performance was acceptable but not remar...
Label  : NEGATIVE
Score  : 0.9953
-------------------------------------------------------


In [21]:
# ---------------------------------------------------------------
# Task 2 : Named Entity Recognition (NER)
# Finds people, organizations, locations, dates in text
# aggregation_strategy='simple' merges subword tokens into words
# ---------------------------------------------------------------

ner = pipeline('ner', aggregation_strategy='simple')

text = (
    'Yann LeCun joined Meta AI in New York after working at '
    'AT&T Bell Labs. He won the Turing Award in 2018 alongside '
    'Geoffrey Hinton and Yoshua Bengio.'
)

entities = ner(text)

print('Named Entity Recognition')
print('=' * 55)
print(f'Input : {text}')
print()
print(f'{"Entity":<30} {"Type":<12} {"Score"}')
print('-' * 55)
for e in entities:
    print(f"{e['word']:<30} {e['entity_group']:<12} {e['score']:.4f}")

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Named Entity Recognition
Input : Yann LeCun joined Meta AI in New York after working at AT&T Bell Labs. He won the Turing Award in 2018 alongside Geoffrey Hinton and Yoshua Bengio.

Entity                         Type         Score
-------------------------------------------------------
Yann LeCun                     PER          0.9963
Meta AI                        ORG          0.9983
New York                       LOC          0.9989
AT & T Bell Labs               ORG          0.9994
Turing Award                   MISC         0.9885
Geoffrey Hinton                PER          0.9997
Yoshua Bengio                  PER          0.9989


In [22]:
# ---------------------------------------------------------------
# Task 3 : Question Answering (Extractive)
# The model does NOT generate an answer — it extracts a span
# from the context. This is called extractive QA.
# ---------------------------------------------------------------

qa = pipeline('question-answering')

context = """
The Transformer architecture was introduced in the 2017 paper
'Attention Is All You Need' by Vaswani et al. at Google Brain.
It replaced recurrent networks with self-attention, allowing
full parallelization during training. BERT was released in 2018
and used a bidirectional encoder trained with masked language
modeling. GPT-3, released in 2020, demonstrated that scaling
decoder-only models to 175 billion parameters produced emergent
capabilities not seen in smaller models.
"""

questions = [
    'Who introduced the Transformer architecture?',
    'What training objective did BERT use?',
    'How many parameters does GPT-3 have?',
]

print('Extractive Question Answering')
print('=' * 55)
for q in questions:
    result = qa(question=q, context=context)
    print(f'Q : {q}')
    print(f'A : {result["answer"]}')
    print(f'   (confidence : {result["score"]:.4f})')
    print()

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.


Extractive Question Answering
Q : Who introduced the Transformer architecture?
A : Vaswani et al. at Google Brain
   (confidence : 0.3645)

Q : What training objective did BERT use?
A : bidirectional encoder trained with masked language
modeling
   (confidence : 0.2268)

Q : How many parameters does GPT-3 have?
A : 175 billion
   (confidence : 0.7870)



In [23]:
# ---------------------------------------------------------------
# Task 4 : Zero-Shot Classification
# No fine-tuning required. Works by framing classification
# as natural language inference (NLI). The model checks whether
# the text entails each candidate label description.
# ---------------------------------------------------------------

zero_shot = pipeline('zero-shot-classification')

texts_and_labels = [
    {
        'text'   : 'The Federal Reserve raised interest rates by 25 basis points.',
        'labels' : ['economics', 'sports', 'technology', 'politics']
    },
    {
        'text'   : 'New self-attention variants reduce quadratic complexity to linear.',
        'labels' : ['machine learning', 'cooking', 'travel', 'finance']
    },
]

print('Zero-Shot Classification')
print('=' * 55)
for item in texts_and_labels:
    result = zero_shot(item['text'], candidate_labels=item['labels'])
    print(f'Text   : {item["text"][:60]}...')
    print('Scores :')
    for label, score in zip(result['labels'], result['scores']):
        bar = '█' * int(score * 30)
        print(f'  {label:<20} {score:.4f}  {bar}')
    print()

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.


Zero-Shot Classification
Text   : The Federal Reserve raised interest rates by 25 basis points...
Scores :
  economics            0.5889  █████████████████
  technology           0.1693  █████
  politics             0.1311  ███
  sports               0.1107  ███

Text   : New self-attention variants reduce quadratic complexity to l...
Scores :
  machine learning     0.3382  ██████████
  travel               0.3179  █████████
  finance              0.1996  █████
  cooking              0.1443  ████



---

## Part 3 — Tokenizers: From Text to Tensors

Every pipeline hides a tokenizer. Understanding tokenizers is
essential because they determine what your model actually sees.
A bug in tokenization silently produces wrong results.

### What a Tokenizer Does

```
Raw text → Tokens → Token IDs → Attention mask → Model input

Step 1  :  Normalize  →  lowercase, strip accents, handle Unicode
Step 2  :  Split      →  break text into subword units
Step 3  :  Map        →  convert each token to its integer ID
Step 4  :  Add specials → [CLS], [SEP], <s>, </s> depend on the model
Step 5  :  Pad/Truncate → make all sequences the same length for batching
Step 6  :  Mask        →  attention_mask = 1 for real tokens, 0 for padding
```

### Why Subword Tokenization?

Character-level: too long sequences, no word-level semantics
Word-level: vocabulary explodes with morphology ('run', 'running', 'runner'...)

```
Subword algorithms (BPE, WordPiece, SentencePiece) balance both:

'unbelievable' → ['un', '##believ', '##able']   (WordPiece / BERT)
'unbelievable' → ['▁un', 'believ', 'able']       (SentencePiece / T5)
'unbelievable' → ['un', 'believ', 'able']        (BPE / GPT)

Common words get single tokens.
Rare or new words are split into familiar subpieces.
Nothing is truly out-of-vocabulary.
```

### Special Tokens by Architecture

```
BERT    :  [CLS] sentence [SEP] sentence [SEP]
           [CLS] always goes first — its output is used for classification

GPT-2   :  sentence  (no special tokens by default, just text)
           Generation stops at <|endoftext|>

T5      :  sentence </s>  (encoder input)
           <pad> token_1 token_2 ... </s>  (decoder input/output)
```

In [24]:
from transformers import AutoTokenizer

# ---------------------------------------------------------------
# Load BERT tokenizer and inspect every step
# AutoTokenizer figures out the right tokenizer class automatically
# ---------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

text = 'Transformers are surprisingly intuitive once you understand attention.'

# Step 1 : See the raw tokens
tokens = tokenizer.tokenize(text)
print('Raw Tokens')
print('=' * 55)
print(tokens)
print(f'\nToken count : {len(tokens)}')

print()

# Step 2 : Convert to IDs — this is what the model sees
ids = tokenizer.convert_tokens_to_ids(tokens)
print('Token IDs (no special tokens yet)')
print('=' * 55)
print(ids)

Raw Tokens
['transformers', 'are', 'surprisingly', 'intuitive', 'once', 'you', 'understand', 'attention', '.']

Token count : 9

Token IDs (no special tokens yet)
[19081, 2024, 10889, 29202, 2320, 2017, 3305, 3086, 1012]


In [25]:
# ---------------------------------------------------------------
# Full encoding pipeline — what the model actually receives
# return_tensors='pt' gives PyTorch tensors
# ---------------------------------------------------------------

encoded = tokenizer(text, return_tensors='pt')

print('Full Encoded Output')
print('=' * 55)
print('Keys in output      :', list(encoded.keys()))
print()
print('input_ids shape     :', encoded['input_ids'].shape)
print('input_ids           :', encoded['input_ids'])
print()
print('attention_mask      :', encoded['attention_mask'])
print('  (1 = real token,   0 = padding — no padding here since single input)')
print()

# Decode back to text
decoded = tokenizer.decode(encoded['input_ids'][0])
print('Decoded back        :', decoded)

Full Encoded Output
Keys in output      : ['input_ids', 'token_type_ids', 'attention_mask']

input_ids shape     : torch.Size([1, 11])
input_ids           : tensor([[  101, 19081,  2024, 10889, 29202,  2320,  2017,  3305,  3086,  1012,
           102]])

attention_mask      : tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
  (1 = real token,   0 = padding — no padding here since single input)

Decoded back        : [CLS] transformers are surprisingly intuitive once you understand attention. [SEP]


In [26]:
# ---------------------------------------------------------------
# Padding and truncation — handling variable-length batches
# In practice you always batch inputs. Padding makes them equal length.
# ---------------------------------------------------------------

sentences = [
    'Short sentence.',
    'This sentence is quite a bit longer and has more tokens in it.',
    'Medium length sentence here.',
]

batch = tokenizer(
    sentences,
    padding    = True,   # pad shorter sequences to match longest
    truncation = True,   # truncate sequences longer than max_length
    max_length = 20,
    return_tensors = 'pt'
)

print('Batched Encoding with Padding')
print('=' * 55)
print('input_ids shape     :', batch['input_ids'].shape)
print('  (batch_size=3, sequence_length=20)')
print()

for i, sentence in enumerate(sentences):
    ids   = batch['input_ids'][i]
    mask  = batch['attention_mask'][i]
    real_tokens = mask.sum().item()
    pad_tokens  = (mask == 0).sum().item()
    print(f'Sentence {i+1} : {sentence[:40]}')
    print(f'  Real tokens : {real_tokens}   Padding tokens : {pad_tokens}')

print()
print('attention_mask (1=real, 0=pad):')
print(batch['attention_mask'])
print()
print('Note: the model ignores positions where attention_mask=0')

Batched Encoding with Padding
input_ids shape     : torch.Size([3, 17])
  (batch_size=3, sequence_length=20)

Sentence 1 : Short sentence.
  Real tokens : 5   Padding tokens : 12
Sentence 2 : This sentence is quite a bit longer and 
  Real tokens : 17   Padding tokens : 0
Sentence 3 : Medium length sentence here.
  Real tokens : 7   Padding tokens : 10

attention_mask (1=real, 0=pad):
tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

Note: the model ignores positions where attention_mask=0


In [27]:
# ---------------------------------------------------------------
# Compare tokenizers across model families
# Same text, completely different tokenization strategies
# ---------------------------------------------------------------

word = 'unbelievable'

models = {
    'BERT (WordPiece)'   : 'bert-base-uncased',
    'GPT-2 (BPE)'        : 'gpt2',
    'RoBERTa (BPE)'      : 'roberta-base',
}

print(f'Tokenization comparison for: "{word}"')
print('=' * 55)
for name, model_id in models.items():
    tok    = AutoTokenizer.from_pretrained(model_id)
    tokens = tok.tokenize(word)
    print(f'{name:<28} → {tokens}')

Tokenization comparison for: "unbelievable"
BERT (WordPiece)             → ['unbelievable']
GPT-2 (BPE)                  → ['un', 'bel', 'iev', 'able']
RoBERTa (BPE)                → ['un', 'bel', 'iev', 'able']


---

## Part 4 — AutoModel: Loading and Running Models Directly

Pipelines are convenient but abstract. For custom tasks you need
to load the model directly and run a forward pass yourself.

### The Auto Classes

```
AutoTokenizer             →  loads the correct tokenizer for any model
AutoModel                 →  base model, outputs raw hidden states
AutoModelForSequenceClassification  →  adds a classification head
AutoModelForTokenClassification     →  adds NER-style per-token head
AutoModelForQuestionAnswering       →  adds start/end span heads
AutoModelForSeq2SeqLM               →  encoder-decoder generation
AutoModelForCausalLM                →  decoder-only generation (GPT)
AutoModelForMaskedLM                →  fill-mask (BERT style)
```

The Auto prefix means: figure out the right class from the model card,
then load it. You do not need to know whether it is BERT, RoBERTa,
DistilBERT, or anything else — the identifier string handles it.

### What the Model Returns

```
outputs = model(**inputs)

outputs.last_hidden_state  →  shape (batch, seq_len, hidden_dim)
                               one vector per token per example

outputs.pooler_output      →  shape (batch, hidden_dim)
                               [CLS] token passed through a linear layer
                               used for sentence-level tasks

outputs.logits             →  shape (batch, num_classes) for classifiers
                               raw scores before softmax
```

In [28]:
from transformers import AutoModel, AutoTokenizer

# ---------------------------------------------------------------
# Load raw BERT and run a forward pass manually
# This gives you full control over what you do with the outputs
# ---------------------------------------------------------------

model_name = 'bert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModel.from_pretrained(model_name)

model.eval()  # disable dropout for inference

text = 'Attention is all you need.'
inputs = tokenizer(text, return_tensors='pt')

with torch.no_grad():       # no_grad saves memory — we are not training
    outputs = model(**inputs)

print('BERT Forward Pass')
print('=' * 55)
print('Input text              :', text)
print('Token count             :', inputs['input_ids'].shape[1])
print()
print('last_hidden_state shape :', outputs.last_hidden_state.shape)
print('  → (batch=1, tokens, hidden_dim=768)')
print()
print('pooler_output shape     :', outputs.pooler_output.shape)
print('  → (batch=1, hidden_dim=768)  — the [CLS] representation')
print()

# Each token gets its own 768-dimensional vector
print('Token representations (first 3 tokens, first 5 dims):')
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
for i in range(3):
    vec = outputs.last_hidden_state[0, i, :5].numpy()
    print(f'  {tokens[i]:<12} → {vec}')

BERT Forward Pass
Input text              : Attention is all you need.
Token count             : 8

last_hidden_state shape : torch.Size([1, 8, 768])
  → (batch=1, tokens, hidden_dim=768)

pooler_output shape     : torch.Size([1, 768])
  → (batch=1, hidden_dim=768)  — the [CLS] representation

Token representations (first 3 tokens, first 5 dims):
  [CLS]        → [ 0.0716886   0.03177569 -0.12116557 -0.16685553 -0.4738068 ]
  attention    → [ 0.46534526  0.20061564 -0.02856164 -0.54736423  0.40286645]
  is           → [-0.02443836 -0.9304197   0.24858351 -0.52673995  0.50112855]


In [29]:
# ---------------------------------------------------------------
# Sentence embeddings from BERT using mean pooling
# Average the token embeddings (ignoring padding) to get
# a single fixed-size vector representing the whole sentence
# ---------------------------------------------------------------

def mean_pooling(model_output, attention_mask):
    """
    Average token embeddings, weighted by attention mask.
    This ignores [PAD] tokens when computing the mean.
    """
    hidden_states = model_output.last_hidden_state  # (batch, seq, dim)
    mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
    sum_embeddings = (hidden_states * mask_expanded).sum(1)  # (batch, dim)
    sum_mask       = mask_expanded.sum(1).clamp(min=1e-9)    # avoid div by zero
    return sum_embeddings / sum_mask


sentences = [
    'Machine learning is a subset of artificial intelligence.',
    'Deep learning uses neural networks with many layers.',
    'I enjoy hiking in the mountains on weekends.',
]

inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

embeddings = mean_pooling(outputs, inputs['attention_mask'])

# Normalize to unit sphere (cosine similarity becomes dot product)
import torch.nn.functional as F
embeddings = F.normalize(embeddings, p=2, dim=1)

# Cosine similarity matrix
sim = torch.mm(embeddings, embeddings.T)

print('Sentence Similarity (cosine)')
print('=' * 55)
short = [s[:35] for s in sentences]
for i, s1 in enumerate(short):
    for j, s2 in enumerate(short):
        if j > i:
            score = sim[i, j].item()
            print(f'{s1[:30]}...')
            print(f'{s2[:30]}...')
            print(f'  Similarity : {score:.4f}')
            print()

print('Sentences 1 and 2 (both AI-related) should be more similar')
print('than either compared to sentence 3 (hiking).')

Sentence Similarity (cosine)
Machine learning is a subset o...
Deep learning uses neural netw...
  Similarity : 0.7958

Machine learning is a subset o...
I enjoy hiking in the mountain...
  Similarity : 0.5241

Deep learning uses neural netw...
I enjoy hiking in the mountain...
  Similarity : 0.5218

Sentences 1 and 2 (both AI-related) should be more similar
than either compared to sentence 3 (hiking).


---

## Part 5 — Reading a Model Card

Every model on the Hub has a Model Card — a documentation page that
tells you everything you need to decide if this model is right for
your use case.

### What to Check in a Model Card

```
1. Model Description
   → What architecture? (BERT, GPT, T5, ...)
   → How many parameters? (base ~110M, large ~340M, 7B, 70B...)
   → What language(s) was it trained on?

2. Training Data
   → What corpus? (Common Crawl, Wikipedia, Books, code...)
   → What time period? (knowledge cutoff matters for factual tasks)
   → Any known biases in the training data?

3. Intended Use
   → What task was it fine-tuned for, if any?
   → Is it a base model (needs fine-tuning) or instruction-tuned?

4. Evaluation Results
   → Benchmark scores: GLUE, SQuAD, MMLU, HumanEval, HellaSwag...
   → Compare against other models for your specific task

5. Limitations and Biases
   → Well-maintained cards document known failure modes
   → What languages or domains does the model struggle with?

6. License
   → MIT, Apache 2.0, CC-BY — permissive, use commercially
   → Llama 2 license — requires Meta approval for >700M users
   → Some models are gated — you need Hub account approval
```

### Choosing the Right Model Size

```
Task                  Recommended size           Why
─────────────────────────────────────────────────────────────────
Classification        distilbert (66M)           Fast, good accuracy
Embeddings            all-MiniLM-L6 (22M)        Best speed/quality trade-off
NER                   bert-base (110M)           Proven, well-understood
Summarization         facebook/bart-large (400M) Strong seq2seq baseline
Chat (local)          Mistral-7B (7B)            Excellent for compute budget
Chat (best quality)   via API (Claude, GPT-4)   When accuracy > cost
```

In [30]:
# ---------------------------------------------------------------
# Inspect model architecture and count parameters
# This is useful when reading a model card — you can verify
# the parameter count and understand the structure
# ---------------------------------------------------------------

def count_parameters(model):
    total   = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


total, trainable = count_parameters(model)

print('BERT-base-uncased Parameter Count')
print('=' * 55)
print(f'Total parameters      : {total:,}')
print(f'Trainable parameters  : {trainable:,}')
print(f'Size (float32)        : {total * 4 / 1e6:.1f} MB')
print()

# Show the top-level module structure
print('Module Structure')
print('=' * 55)
for name, module in model.named_children():
    params = sum(p.numel() for p in module.parameters())
    print(f'{name:<20} {type(module).__name__:<25} {params:>12,} params')

print()
print('The encoder contains 12 identical Transformer blocks.')
print('Each block has self-attention + feed-forward + layer norm.')

BERT-base-uncased Parameter Count
Total parameters      : 109,482,240
Trainable parameters  : 109,482,240
Size (float32)        : 437.9 MB

Module Structure
embeddings           BertEmbeddings              23,837,184 params
encoder              BertEncoder                 85,054,464 params
pooler               BertPooler                     590,592 params

The encoder contains 12 identical Transformer blocks.
Each block has self-attention + feed-forward + layer norm.


---

## Day 7 Summary

```
What you built today:

1.  Pipelines           →  sentiment, NER, QA, zero-shot in one line each
2.  Tokenizers          →  text → tokens → IDs → padded batches
3.  AutoModel           →  raw forward pass, hidden states, embeddings
4.  Parameter counting  →  inspect any model's architecture and size
5.  Model Card reading  →  how to evaluate a model before using it

The connection to Day 6:

  The Transformer you built from scratch on Day 6 is exactly what
  lives inside every model you used today. The pipeline is just a
  wrapper around: tokenizer → model.forward() → decode output.

  outputs.last_hidden_state  =  the final layer's token representations
  outputs.pooler_output      =  [CLS] token through one linear layer

What comes next:

  Day 8 covers the most important skill in applied NLP:
  fine-tuning — taking a pretrained model and adapting it to
  your specific task and domain with your own labeled data.
```

### Self-Check Questions

Answer these before Day 8:

1. Why does BERT add a [CLS] token at the start of every input?
2. What does `attention_mask=0` tell the model?
3. Why does GPT-2 use BPE while BERT uses WordPiece?
4. When would you use `AutoModel` instead of a pipeline?
5. What is the difference between extractive QA and generative QA?